# RPS PreTrained YOLO Fine-tuning

## g-drive 마운트

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## 경로 설정

In [ ]:
# 다른 경로에 저장하려면, 아래 경로를 수정하세요
!pip install roboflow

from roboflow import Roboflow
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)
project = rf.workspace("roboflow-58fyf").project("rock-paper-scissors-sxsw")
version = project.version(14)
dataset = version.download("yolov11")

dataset_root = dataset.location  # 예: /content/rock-paper-scissors-sxsw-14
project_root = '/content/rps_runs'
save_dir = '/content/drive/MyDrive/AI_MachineLearning/ai/files/save'



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to rock-paper-scissors-14 in yolov11:: 100%|██████████| 14682/14682 [00:02<00:00, 6554.23it/s]


## RPS 데이터셋 준비

In [ ]:
!unzip -q -o {dataset_zip}
# !tar -xf {dataset_tarball}

## RPS 데이터셋의 Label 만들기

In [ ]:
# 클래스 이름 순서 (data.yaml과 반드시 일치해야 함)
CLASSES = ['scissors', 'rock', 'paper']

In [ ]:
import xml.etree.ElementTree as ET
import os

def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_folder(src_folder, dst_labels_folder):
    """
    src_folder 내의 xml 파일들을 읽어 dst_labels_folder에 txt로 저장
    """
    if not os.path.exists(dst_labels_folder):
        os.makedirs(dst_labels_folder)

    xml_files = [f for f in os.listdir(src_folder) if f.endswith('.xml')]

    for filename in xml_files:
        in_file_path = os.path.join(src_folder, filename)
        out_file_path = os.path.join(dst_labels_folder, filename.replace('.xml', '.txt'))

        tree = ET.parse(in_file_path)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        with open(out_file_path, 'w') as out_file:
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in CLASSES:
                    continue
                cls_id = CLASSES.index(cls)

                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text),
                     float(xmlbox.find('ymin').text),
                     float(xmlbox.find('xmax').text),
                     float(xmlbox.find('ymax').text))

                bb = convert((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")

    print(f"변환 완료: {src_folder} -> {dst_labels_folder} ({len(xml_files)}개)")

# 1. Train 변환 (images/train 폴더 안의 xml -> labels/train 폴더로)
convert_folder(
    src_folder=os.path.join(dataset_root, "images/train"),
    dst_labels_folder=os.path.join(dataset_root, "labels/train")
)

# 2. Test 변환 (images/test 폴더 안의 xml -> labels/test 폴더로)
convert_folder(
    src_folder=os.path.join(dataset_root, "images/test"),
    dst_labels_folder=os.path.join(dataset_root, "labels/test")
)

변환 완료: /content/RPS_Dataset_YOLO/images/train -> /content/RPS_Dataset_YOLO/labels/train (82개)
변환 완료: /content/RPS_Dataset_YOLO/images/test -> /content/RPS_Dataset_YOLO/labels/test (22개)


## Ultralytics 설치 및 import

In [ ]:
!pip -q install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 7.8 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


## 학습

In [ ]:
yolo = YOLO('yolo11n.pt')

In [ ]:
DATA = f'{dataset_root}/data.yaml'

In [ ]:
yolo.train(data=DATA, epochs=80, batch=32,
           imgsz=320,
           device=0,
           workers=4,
           project=project_root,
           degrees=15.0,
           flipud=0.5,
           fliplr=0.5,
           mosaic=1.0,
           name='rps_yolo11n')

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rps_yolo11n, nbs=64, nms

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e197a0b04b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

## ONNX 모델로 변환

In [ ]:
BEST = f'{project_root}/rps_yolo11n/weights/best.pt'

# model obejct instantiation w/ best.pt
yolo_best = YOLO(BEST)
# model export to onnx format
yolo_best.export(format='onnx',
                 imgsz=320,
                 #int8=True,
                 opset=18,
                 simplify=True)


Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 316ms
Prepared 4 packages in 1.91s
Installed 4 packages in 257ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 3.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming

'/content/rps_runs/rps_yolo11n/weights/best.onnx'

## ONNX 파일 저장

In [ ]:
onnx_best = 'rps_yolo11n.onnx'
save_file = f'{save_dir}/{onnx_best}'

In [ ]:
!cp {project_root}/rps_yolo11n/weights/best.onnx {onnx_best}

In [ ]:
!cp {onnx_best} {save_file}

In [ ]:
from google.colab import files
files.download(onnx_best)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>